# Эксперимент 18 v4: SSL Domain Adaptation — LightlySSL DINO (self-distillation)

## Отличия от v1/v2 (SimCLR)

| | v1/v2 (SimCLR) | **v4 (DINO)** |
|---|---|---|
| Метод | NT-Xent, нужны негативы | Self-distillation, негативы не нужны |
| Архитектура | backbone + projection head | Student + Teacher (EMA-копия студента) |
| Зависимость от batch | Критична (мало негативов = слабый сигнал) | Минимальна — работает при batch=4 |
| Projection head | MLP 768→512→128 | DINOProjectionHead 768→512→64→2048 |
| Loss | NT-Xent | DINO cross-entropy с centering + sharpening |
| Обновление teacher | — | EMA с косинусным расписанием momentum |

## Пайплайн
```
DINOv2 (pretrained) → SSL DINO 60 эп. (12568 изобр. без разметки)
    → DINOv2* (адаптирован к домену стали)
    → Fine-tune Эксп.17 (778 реал. + 2269 синт., CE+Lovász, AMP, layer-wise LR)
    → Оценка mIoU на val (1334 изобр.)
```

## 1. Импорты и конфигурация

In [1]:
# !pip install lightly  # раскомментировать если lightly не установлен

import os, random, json, pickle, copy, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchmetrics import JaccardIndex
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

from lightly.loss import DINOLoss
from lightly.models.modules import DINOProjectionHead
from lightly.models.utils import deactivate_requires_grad, update_momentum

print(f'PyTorch: {torch.__version__}')
import lightly; print(f'Lightly: {lightly.__version__}')

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Устройство: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} Гб')

DATA_DIR      = 'D:/VKR/VKR/DATA_DIR/'
TRAIN_CSV     = os.path.join(DATA_DIR, 'train.csv')
TRAIN_IMG_DIR = os.path.join(DATA_DIR, 'train_images/')
save_dir      = 'D:/VKR/VKR/dino_heads'
os.makedirs(save_dir, exist_ok=True)

# ── Архитектура ───────────────────────────────────────────────────────
IMG_H       = 224
IMG_W       = 1400
PATCH_SIZE  = 14
PATCH_H     = IMG_H // PATCH_SIZE   # 16
PATCH_W     = IMG_W // PATCH_SIZE   # 100
MASK_H      = PATCH_H * 4          # 64
MASK_W      = PATCH_W * 4          # 400
EMBED_DIM   = 768
NUM_CLASSES = 5
INTERMEDIATE_LAYERS = [3, 5, 8, 11]

# ── Данные (Эксп.13) ──────────────────────────────────────────────────
N_PER_CLASS  = 200
TEST_SIZE    = 0.2
SYNTH_WEIGHT = 0.5

# ── Fine-tuning (Эксп.17) ─────────────────────────────────────────────
N_UNFREEZE  = 4
LR_BACKBONE = 1e-5
LR_DECAY    = 0.75
LR_HEAD     = 1e-3
BATCH_SIZE  = 4
EPOCHS      = 75
FLIP_P      = 0.5
CROP_SCALE  = (0.85, 1.0)
BRIGHTNESS  = 0.3
USE_AMP     = (DEVICE == 'cuda')

# ── SSL DINO (v4: LightlySSL self-distillation) ───────────────────────
SSL_BATCH_SIZE               = 4
SSL_EPOCHS                   = 60
SSL_LR_BACKBONE              = 1e-5
SSL_LR_HEAD                  = 1e-3
SSL_MOMENTUM_TEACHER_START   = 0.996   # EMA momentum (по косинусу растёт до 1.0)
DINO_OUT_DIM                 = 2048
DINO_HIDDEN_DIM              = 512
DINO_BOTTLENECK_DIM          = 64
DINO_WARMUP_TEACHER_TEMP_EP  = 5       # эпох для прогрева температуры учителя
DINO_TEACHER_TEMP            = 0.07
DINO_STUDENT_TEMP            = 0.1
SSL_SAVE_LABEL               = 'exp18v4'

print(f'SSL batch: {SSL_BATCH_SIZE}  |  SSL epochs: {SSL_EPOCHS}')
print(f'DINO out_dim: {DINO_OUT_DIM}  |  momentum start: {SSL_MOMENTUM_TEACHER_START}')
print(f'Fine-tune batch: {BATCH_SIZE} | AMP: {USE_AMP}')

PyTorch: 2.12.0.dev20260408+cu128
Lightly: 1.5.24
Устройство: cuda
GPU: NVIDIA GeForce RTX 5060 Laptop GPU
VRAM: 8.5 Гб
SSL batch: 4  |  SSL epochs: 60
DINO out_dim: 2048  |  momentum start: 0.996
Fine-tune batch: 4 | AMP: True


## 2. Загрузка данных

In [2]:
train_df    = pd.read_csv(TRAIN_CSV)
labeled_ids = train_df['ImageId'].unique().tolist()
print(f'Строк: {len(train_df):,}  |  Размеченных изображений: {len(labeled_ids):,}')

# Все изображения в папке (включая фоновые без аннотаций) — для SSL
all_train_imgs = sorted([f for f in os.listdir(TRAIN_IMG_DIR) if f.endswith('.jpg')])
print(f'Всего изображений в train_images: {len(all_train_imgs):,}  (используем для SSL)')


def decode_rle(rle_string, shape=(256, 1600)):
    if pd.isna(rle_string) or not isinstance(rle_string, str):
        return np.zeros(shape, dtype=np.uint8)
    nums   = list(map(int, rle_string.strip().split()))
    starts = np.array(nums[0::2]) - 1
    lens   = np.array(nums[1::2])
    mask   = np.zeros(shape[0]*shape[1], dtype=np.uint8)
    for s, l in zip(starts, lens):
        mask[s:s+l] = 1
    return mask.reshape(shape, order='F')


def build_segmask(image_id, df, shape=(256, 1600)):
    mask = np.zeros(shape, dtype=np.uint8)
    for _, row in df[df['ImageId'] == image_id].iterrows():
        cls = int(row['ClassId'])
        m   = decode_rle(row['EncodedPixels'], shape)
        mask[m == 1] = cls
    return mask


def compute_class_weights(image_ids, df, num_classes=NUM_CLASSES):
    px = Counter({c: 0 for c in range(num_classes)})
    for img_id in image_ids:
        mask = build_segmask(img_id, df)
        for c in range(num_classes):
            px[c] += int((mask == c).sum())
    total = sum(px.values())
    w = torch.tensor([total/(num_classes*(px[c]+1e-6)) for c in range(num_classes)])
    w = (w / w.mean()).clamp(min=0.1, max=5.0)
    for c, v in enumerate(w):
        print(f'  {"Фон" if c==0 else f"Дефект {c}"}: {v:.3f}  ({px[c]:,} пикс.)')
    return w.to(DEVICE)


classes_cache = train_df.groupby('ImageId')['ClassId'].apply(
    lambda x: sorted(x.dropna().astype(int).unique().tolist())
).to_dict()
label_map = {img_id: (cls_list[0] if cls_list else 0)
             for img_id, cls_list in classes_cache.items()}

print('Вспомогательные функции определены.')

Строк: 7,095  |  Размеченных изображений: 6,666
Всего изображений в train_images: 12,568  (используем для SSL)
Вспомогательные функции определены.


## 3. Dataset (fine-tuning)

In [3]:
class JointTransform:
    def __init__(self, img_h=IMG_H, img_w=IMG_W, is_train=True,
                 flip_p=FLIP_P, crop_scale=CROP_SCALE, brightness=BRIGHTNESS):
        self.img_h        = img_h
        self.img_w        = img_w
        self.is_train     = is_train
        self.flip_p       = flip_p
        self.crop_scale   = crop_scale
        self.color_jitter = transforms.ColorJitter(brightness=brightness)
        self.to_tensor    = transforms.ToTensor()
        self.normalize    = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __call__(self, img_pil, mask_np):
        img_pil  = img_pil.resize((self.img_w, self.img_h), Image.BILINEAR)
        mask_pil = Image.fromarray(mask_np).resize(
            (self.img_w, self.img_h), Image.NEAREST)
        if self.is_train:
            if random.random() < self.flip_p:
                img_pil  = img_pil.transpose(Image.FLIP_LEFT_RIGHT)
                mask_pil = mask_pil.transpose(Image.FLIP_LEFT_RIGHT)
            scale  = random.uniform(*self.crop_scale)
            crop_h = max(1, int(self.img_h * scale))
            crop_w = max(1, int(self.img_w * scale))
            top    = random.randint(0, self.img_h - crop_h)
            left   = random.randint(0, self.img_w - crop_w)
            img_pil  = img_pil.crop((left, top, left+crop_w, top+crop_h))
            mask_pil = mask_pil.crop((left, top, left+crop_w, top+crop_h))
            img_pil  = img_pil.resize((self.img_w, self.img_h), Image.BILINEAR)
            mask_pil = mask_pil.resize((self.img_w, self.img_h), Image.NEAREST)
            img_pil  = self.color_jitter(img_pil)
        img_t    = self.normalize(self.to_tensor(img_pil))
        mask_np2 = np.array(mask_pil, dtype=np.uint8)
        return img_t, mask_np2


class SteelSegDataset(Dataset):
    def __init__(self, ids, img_dir, df, joint_transform):
        self.ids=ids; self.img_dir=img_dir; self.df=df; self.jt=joint_transform
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        img_id = self.ids[idx]
        img    = Image.open(os.path.join(self.img_dir, img_id)).convert('RGB')
        mask   = build_segmask(img_id, self.df)
        img_t, mask_np = self.jt(img, mask)
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        mask_t = F.interpolate(
            mask_t.unsqueeze(0), size=(MASK_H, MASK_W),
            mode='nearest').squeeze().long()
        return img_t, mask_t, 0


class SteelSegDatasetWithSynth(Dataset):
    def __init__(self, real_ids, img_dir, df, joint_transform, synth_pairs):
        self.real_ids = real_ids; self.img_dir = img_dir; self.df = df
        self.jt = joint_transform; self.synth = synth_pairs
        self.n_real = len(real_ids); self.n_synth = len(synth_pairs)
    def __len__(self): return self.n_real + self.n_synth
    def __getitem__(self, idx):
        if idx < self.n_real:
            img_id   = self.real_ids[idx]
            img      = Image.open(os.path.join(self.img_dir, img_id)).convert('RGB')
            mask     = build_segmask(img_id, self.df)
            is_synth = 0
        else:
            img_np, mask = self.synth[idx - self.n_real]
            img          = Image.fromarray(img_np.astype(np.uint8))
            is_synth     = 1
        img_t, mask_np = self.jt(img, mask)
        mask_t = torch.from_numpy(mask_np).unsqueeze(0).float()
        mask_t = F.interpolate(
            mask_t.unsqueeze(0), size=(MASK_H, MASK_W),
            mode='nearest').squeeze().long()
        return img_t, mask_t, is_synth


train_jt = JointTransform(is_train=True)
val_jt   = JointTransform(is_train=False)
print('Datasets определены.')

Datasets определены.


## 4. Загрузка DINOv2

In [4]:
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vitb14',
                        pretrained=True, verbose=False)
dinov2 = dinov2.to(DEVICE)
for p in dinov2.parameters():
    p.requires_grad = False

n_blocks = len(dinov2.blocks)   # 12
total_p  = sum(p.numel() for p in dinov2.parameters())
print(f'DINOv2 ViT-B/14: {total_p/1e6:.1f}М параметров, {n_blocks} блоков')
print(f'Все параметры заморожены до SSL-стадии.')

C:\Users\MSI Katana 17/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
C:\Users\MSI Katana 17/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
C:\Users\MSI Katana 17/.cache\torch\hub\facebookresearch_dinov2_main\dinov2\layers\block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


DINOv2 ViT-B/14: 86.6М параметров, 12 блоков
Все параметры заморожены до SSL-стадии.


## 5. Эмбеддинги (из кэша)

In [5]:
emb_cache = os.path.join(save_dir, 'embeddings.pt')
print('Загружаем из кэша...')
ckpt          = torch.load(emb_cache, map_location='cpu')
embeddings    = ckpt['embeddings']
emb_ids       = ckpt['emb_ids']
emb_id_to_idx = {v: k for k, v in enumerate(emb_ids)}
print(f'Загружено: {embeddings.shape}')

Загружаем из кэша...
Загружено: torch.Size([6666, 768])


## 6. KMeans-отбор (200 на класс → ~778 реальных)

In [6]:
all_labels = [label_map.get(i, 0) for i in labeled_ids]
train_ids, test_ids = train_test_split(
    labeled_ids, test_size=TEST_SIZE, stratify=all_labels, random_state=SEED)
print(f'Train pool: {len(train_ids):,}  |  Test: {len(test_ids):,}')


def kmeans_select(embeddings, image_ids, n_select, seed=SEED):
    km = KMeans(n_clusters=n_select, random_state=seed, n_init=10)
    km.fit(embeddings.numpy())
    selected = []
    for k in range(n_select):
        m = km.labels_ == k
        if not m.any(): continue
        center  = torch.tensor(km.cluster_centers_[k])
        ix      = np.where(m)[0]
        nearest = int(ix[torch.norm(embeddings[m] - center, dim=1).argmin().item()])
        selected.append(image_ids[nearest])
    return selected


def select_per_class(embeddings, image_ids, classes_cache,
                     n_per_class=N_PER_CLASS, seed=SEED):
    image_id_to_idx = {v: k for k, v in enumerate(image_ids)}
    selected_set    = set()
    for cls in [1, 2, 3, 4]:
        cls_ids = [i for i in image_ids if cls in classes_cache.get(i, [])]
        n_avail = len(cls_ids)
        if n_avail == 0: continue
        if n_avail <= n_per_class:
            chosen = cls_ids
            print(f'  Класс {cls}: {n_avail} доступно → берём все')
        else:
            idxs       = [image_id_to_idx[i] for i in cls_ids]
            cls_embeds = embeddings[idxs]
            chosen     = kmeans_select(cls_embeds, cls_ids, n_per_class, seed)
            print(f'  Класс {cls}: {n_avail} → {len(chosen)} (KMeans)')
        selected_set.update(chosen)
    result = list(selected_set)
    print(f'Итого уникальных: {len(result)}')
    return result


pool_mask    = [emb_id_to_idx[i] for i in train_ids]
pool_embeds  = embeddings[pool_mask]
print('Отбор 200 изображений на класс:')
selected_ids = select_per_class(pool_embeds, train_ids, classes_cache)
assert len(set(selected_ids) & set(test_ids)) == 0, 'Пересечение с тест-сетом!'
print('Пересечений с тест-сетом: 0 ✓')

Train pool: 5,332  |  Test: 1,334
Отбор 200 изображений на класс:
  Класс 1: 717 → 200 (KMeans)
  Класс 2: 197 доступно → берём все
  Класс 3: 4113 → 200 (KMeans)
  Класс 4: 641 → 200 (KMeans)
Итого уникальных: 778
Пересечений с тест-сетом: 0 ✓


## 7. Архитектура: SegHeadDPT + Lovász-Softmax

In [7]:
class SegHeadDPT(nn.Module):
    def __init__(self, embed_dim=EMBED_DIM, num_classes=NUM_CLASSES,
                 patch_h=PATCH_H, patch_w=PATCH_W, n_layers=4):
        super().__init__()
        self.patch_h  = patch_h
        self.patch_w  = patch_w
        self.n_layers = n_layers
        self.proj = nn.ModuleList([
            nn.Sequential(nn.Conv2d(embed_dim, 256, 1),
                          nn.BatchNorm2d(256), nn.GELU())
            for _ in range(n_layers)
        ])
        self.fuse = nn.Sequential(
            nn.Conv2d(256 * n_layers, 512, 1), nn.BatchNorm2d(512), nn.GELU(),
            nn.Conv2d(512, 256, 3, padding=1), nn.BatchNorm2d(256), nn.GELU())
        self.up1 = nn.Sequential(
            nn.ConvTranspose2d(256, 128, 2, stride=2), nn.BatchNorm2d(128), nn.GELU(),
            nn.Conv2d(128, 128, 3, padding=1), nn.BatchNorm2d(128), nn.GELU())
        self.up2 = nn.Sequential(
            nn.ConvTranspose2d(128, 64, 2, stride=2), nn.BatchNorm2d(64), nn.GELU(),
            nn.Conv2d(64, 64, 3, padding=1), nn.BatchNorm2d(64), nn.GELU())
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, features):
        maps = []
        for i, f in enumerate(features):
            B, N, C = f.shape
            x = f.reshape(B, self.patch_h, self.patch_w, C).permute(0,3,1,2)
            maps.append(self.proj[i](x))
        x = self.fuse(torch.cat(maps, dim=1))
        return self.head(self.up2(self.up1(x)))


def _lovász_grad(gt_sorted):
    n   = len(gt_sorted)
    gts = gt_sorted.sum()
    intersection = gts - gt_sorted.float().cumsum(0)
    union        = gts + (1.0 - gt_sorted).float().cumsum(0)
    jaccard      = 1.0 - intersection / union
    if n > 1:
        jaccard[1:] = jaccard[1:] - jaccard[:-1]
    return jaccard


def _lovász_softmax_flat(probas, labels):
    C = probas.size(1)
    losses = []
    for c in range(C):
        fg = (labels == c).float()
        if fg.sum() == 0: continue
        errors = (fg - probas[:, c]).abs()
        errors_sorted, perm = torch.sort(errors, descending=True)
        losses.append(torch.dot(errors_sorted, _lovász_grad(fg[perm])))
    return torch.stack(losses).mean() if losses else probas.sum() * 0.0


class LovászSoftmax(nn.Module):
    def forward(self, logits, targets):
        B, C, H, W = logits.shape
        probas  = F.softmax(logits.float(), dim=1).permute(0,2,3,1).reshape(-1, C)
        targets = targets.reshape(-1)
        return _lovász_softmax_flat(probas, targets)


_h = SegHeadDPT().to(DEVICE)
print(f'SegHeadDPT: {sum(p.numel() for p in _h.parameters()):,} параметров')
del _h

SegHeadDPT: 2,845,381 параметров


In [8]:
synth_cache = os.path.join(save_dir, 'synth_pairs_exp13.pkl')
print('Загружаем синтетику...')
with open(synth_cache, 'rb') as f:
    synth_pairs = pickle.load(f)
print(f'Загружено: {len(synth_pairs)} пар  |  Итого: {len(selected_ids)+len(synth_pairs)}')

Загружаем синтетику...
Загружено: 2269 пар  |  Итого: 3047


## 8. Layer-wise LR decay

In [9]:
def make_backbone_param_groups(model, base_lr=LR_BACKBONE, decay=LR_DECAY,
                                n_unfreeze=N_UNFREEZE, weight_decay=1e-2):
    nb = len(model.blocks)
    unfreeze_from = nb - n_unfreeze
    groups = []
    norm_params = [p for p in model.norm.parameters() if p.requires_grad]
    if norm_params:
        groups.append({'params': norm_params, 'lr': base_lr,
                       'weight_decay': weight_decay, 'name': 'norm'})
    for i in range(nb - 1, unfreeze_from - 1, -1):
        block_params = [p for p in model.blocks[i].parameters() if p.requires_grad]
        if not block_params: continue
        depth = nb - 1 - i
        lr_i  = base_lr * (decay ** depth)
        groups.append({'params': block_params, 'lr': lr_i,
                       'weight_decay': weight_decay, 'name': f'block_{i}'})
    return groups


print('Layer-wise LR для fine-tuning (будет применён после SSL):')
for g in make_backbone_param_groups(
        dinov2,  # просто для демонстрации
        base_lr=LR_BACKBONE, decay=LR_DECAY, n_unfreeze=N_UNFREEZE):
    n_p = sum(p.numel() for p in g['params'])
    print(f'  {g["name"]}: lr={g["lr"]:.2e}  ({n_p/1e6:.1f}М)')

Layer-wise LR для fine-tuning (будет применён после SSL):


## 9. SSL Phase — LightlySSL DINO (self-distillation)

### Принцип работы
- **Student** обрабатывает два аугментированных вида → projection head → p(x)
- **Teacher** — EMA-копия студента, обрабатывает те же виды → q(x)
- **Loss**: student учится предсказывать выход учителя: `H(q(v1), p(v2)) + H(q(v2), p(v1))`
- Teacher обновляется только через EMA, никогда через градиент
- Centering (скользящее среднее выходов учителя) предотвращает коллапс без негативных примеров

In [10]:
class DINOAugmentation:
    """Два глобальных вида для DINO — с учётом соотношения сторон стали 1:6.25.
    Нет вертикального отражения и ротации (горизонтальная структура дефектов)."""
    def __init__(self, img_h=IMG_H, img_w=IMG_W):
        self.img_h = img_h
        self.img_w = img_w
        self.color_jitter = transforms.ColorJitter(
            brightness=0.4, contrast=0.4, saturation=0.2, hue=0.1)
        self.blur     = transforms.GaussianBlur(kernel_size=7, sigma=(0.1, 2.0))
        self.to_tensor = transforms.ToTensor()
        self.normalize = transforms.Normalize(
            mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

    def __call__(self, img_pil):
        img = img_pil.resize((self.img_w, self.img_h), Image.BILINEAR)
        # Горизонтальный флип (50%)
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        # Случайный кроп (75–100% площади, сохраняем соотношение сторон)
        scale  = random.uniform(0.75, 1.0)
        crop_h = max(1, int(self.img_h * scale))
        crop_w = max(1, int(self.img_w * scale))
        top    = random.randint(0, self.img_h - crop_h)
        left   = random.randint(0, self.img_w - crop_w)
        img    = img.crop((left, top, left+crop_w, top+crop_h))
        img    = img.resize((self.img_w, self.img_h), Image.BILINEAR)
        # Color Jitter (80%)
        if random.random() < 0.8:
            img = self.color_jitter(img)
        # Grayscale (20%)
        if random.random() < 0.2:
            img = img.convert('L').convert('RGB')
        # Gaussian Blur (50%)
        if random.random() < 0.5:
            img = self.blur(img)
        return self.normalize(self.to_tensor(img))


class DINOSSLDataset(Dataset):
    """Возвращает два независимо аугментированных вида одного изображения."""
    def __init__(self, image_ids, img_dir, augmentation):
        self.image_ids = image_ids
        self.img_dir   = img_dir
        self.aug       = augmentation

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, self.image_ids[idx])
        img      = Image.open(img_path).convert('RGB')
        return self.aug(img), self.aug(img)   # view1, view2


dino_aug = DINOAugmentation()
print('DINOAugmentation и DINOSSLDataset определены.')

DINOAugmentation и DINOSSLDataset определены.


In [11]:
class DINOSSLWrapper(nn.Module):
    """Student + Teacher backbones + projection heads для DINO."""
    def __init__(self, backbone, embed_dim=EMBED_DIM,
                 hidden_dim=DINO_HIDDEN_DIM,
                 bottleneck_dim=DINO_BOTTLENECK_DIM,
                 out_dim=DINO_OUT_DIM):
        super().__init__()
        # Student
        self.student_backbone = backbone
        self.student_head = DINOProjectionHead(
            embed_dim, hidden_dim, bottleneck_dim, out_dim, freeze_last_layer=1)
        # Teacher (EMA-копия, без градиентов)
        self.teacher_backbone = copy.deepcopy(backbone)
        self.teacher_head     = DINOProjectionHead(
            embed_dim, hidden_dim, bottleneck_dim, out_dim)
        deactivate_requires_grad(self.teacher_backbone)
        deactivate_requires_grad(self.teacher_head)
        # Инициализируем веса учителя как у студента
        self.teacher_head.load_state_dict(self.student_head.state_dict())

    def forward_student(self, x):
        cls = self.student_backbone(x)   # (B, 768) CLS-токен
        return self.student_head(cls)

    @torch.no_grad()
    def forward_teacher(self, x):
        cls = self.teacher_backbone(x)
        return self.teacher_head(cls)


def cosine_momentum_schedule(base_momentum, epoch, total_epochs):
    """Плавно увеличивает momentum от base до 1.0 по косинусу."""
    return 1.0 - (1.0 - base_momentum) * (
        np.cos(np.pi * epoch / total_epochs) + 1) / 2


print('DINOSSLWrapper определён.')

DINOSSLWrapper определён.


In [12]:
def train_ssl_dino(all_image_ids, img_dir,
                   n_epochs=SSL_EPOCHS,
                   batch_size=SSL_BATCH_SIZE,
                   label=SSL_SAVE_LABEL):
    """SSL фаза: DINO self-distillation через LightlySSL.
    Возвращает student backbone (адаптированный к домену стали)."""

    print(f'DINO SSL: {n_epochs} эп., batch={batch_size}')
    print(f'  Изображений:      {len(all_image_ids):,}')
    print(f'  Out dim:          {DINO_OUT_DIM}')
    print(f'  Teacher momentum: {SSL_MOMENTUM_TEACHER_START} → 1.0 (косинус)')
    print(f'  Teacher temp warmup: {DINO_WARMUP_TEACHER_TEMP_EP} эп.')

    # ── Backbone студента (разморозка блоков 8-11 + norm) ─────────────
    bb_student = copy.deepcopy(dinov2).to(DEVICE)
    for p in bb_student.parameters():   p.requires_grad = False
    for i in range(n_blocks - N_UNFREEZE, n_blocks):
        for p in bb_student.blocks[i].parameters():  p.requires_grad = True
    for p in bb_student.norm.parameters():           p.requires_grad = True

    frozen_p    = sum(p.numel() for p in bb_student.parameters() if not p.requires_grad)
    trainable_p = sum(p.numel() for p in bb_student.parameters() if p.requires_grad)
    print(f'  Backbone заморожено: {frozen_p/1e6:.1f}М  |  обучается: {trainable_p/1e6:.1f}М')

    # ── DINO model ────────────────────────────────────────────────────
    dino_model = DINOSSLWrapper(bb_student).to(DEVICE)

    # ── Loss ──────────────────────────────────────────────────────────
    criterion = DINOLoss(
        output_dim=DINO_OUT_DIM,
        warmup_teacher_temp=0.04,
        teacher_temp=DINO_TEACHER_TEMP,
        warmup_teacher_temp_epochs=DINO_WARMUP_TEACHER_TEMP_EP,
        student_temp=DINO_STUDENT_TEMP,
        center_momentum=0.9,
    ).to(DEVICE)

    # ── Оптимизатор ───────────────────────────────────────────────────
    backbone_params = [p for p in dino_model.student_backbone.parameters()
                       if p.requires_grad]
    optimizer = optim.AdamW([
        {'params': backbone_params,
         'lr': SSL_LR_BACKBONE, 'weight_decay': 1e-2},
        {'params': list(dino_model.student_head.parameters()),
         'lr': SSL_LR_HEAD,     'weight_decay': 1e-4},
    ])
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=n_epochs)

    # ── DataLoader ────────────────────────────────────────────────────
    ssl_ds = DINOSSLDataset(all_image_ids, img_dir, dino_aug)
    ssl_dl = DataLoader(ssl_ds, batch_size=batch_size,
                        shuffle=True, num_workers=0, drop_last=True)
    print(f'  Батчей/эпоху: {len(ssl_dl)}')

    # ── Checkpoint-resume ─────────────────────────────────────────────
    ckpt_path = os.path.join(save_dir, f'ssl_checkpoint_{label}.pt')
    history   = {'ssl_loss': [], 'momentum': []}
    start_ep  = 0

    if os.path.exists(ckpt_path):
        print(f'  Найден чекпоинт: {ckpt_path} — продолжаем обучение')
        ckpt_data = torch.load(ckpt_path, map_location=DEVICE)
        dino_model.student_backbone.load_state_dict(ckpt_data['student_backbone'])
        dino_model.teacher_backbone.load_state_dict(ckpt_data['teacher_backbone'])
        dino_model.student_head.load_state_dict(ckpt_data['student_head'])
        dino_model.teacher_head.load_state_dict(ckpt_data['teacher_head'])
        optimizer.load_state_dict(ckpt_data['optimizer'])
        scheduler.load_state_dict(ckpt_data['scheduler'])
        criterion.center = ckpt_data['criterion_center'].to(DEVICE)
        history  = ckpt_data['history']
        start_ep = ckpt_data['epoch']
        print(f'  Resume с эпохи {start_ep+1}')

    # ── Обучение ──────────────────────────────────────────────────────
    for epoch in range(start_ep, n_epochs):
        dino_model.student_backbone.train()
        dino_model.student_head.train()
        dino_model.teacher_backbone.eval()
        dino_model.teacher_head.eval()

        momentum = cosine_momentum_schedule(
            SSL_MOMENTUM_TEACHER_START, epoch, n_epochs)

        epoch_losses = []
        for view1, view2 in ssl_dl:
            view1 = view1.to(DEVICE)
            view2 = view2.to(DEVICE)

            # Teacher forward (два вида, без градиента)
            t_out1 = dino_model.forward_teacher(view1)  # (B, DINO_OUT_DIM)
            t_out2 = dino_model.forward_teacher(view2)

            # Student forward (два вида)
            s_out1 = dino_model.forward_student(view1)
            s_out2 = dino_model.forward_student(view2)

            # DINO loss: symmetric — teacher видит оба вида, student тоже
            loss = criterion(
                teacher_out=[t_out1, t_out2],
                student_out=[s_out1, s_out2],
                epoch=epoch)

            optimizer.zero_grad()
            loss.backward()
            # Замораживаем последний слой head на первой эпохе (стабильность)
            dino_model.student_head.cancel_last_layer_gradients(current_epoch=epoch)
            optimizer.step()

            # EMA-обновление teacher
            update_momentum(dino_model.student_backbone,
                            dino_model.teacher_backbone, m=momentum)
            update_momentum(dino_model.student_head,
                            dino_model.teacher_head,     m=momentum)

            epoch_losses.append(loss.item())

        avg_loss = float(np.mean(epoch_losses))
        history['ssl_loss'].append(avg_loss)
        history['momentum'].append(float(momentum))
        scheduler.step()

        if (epoch + 1) % 5 == 0 or epoch == 0:
            print(f'Эп.{epoch+1:3d}/{n_epochs} | '
                  f'DINO Loss: {avg_loss:.4f} | momentum: {momentum:.5f}')

        # Сохраняем чекпоинт каждую эпоху
        torch.save({
            'epoch':            epoch + 1,
            'student_backbone': dino_model.student_backbone.state_dict(),
            'teacher_backbone': dino_model.teacher_backbone.state_dict(),
            'student_head':     dino_model.student_head.state_dict(),
            'teacher_head':     dino_model.teacher_head.state_dict(),
            'optimizer':        optimizer.state_dict(),
            'scheduler':        scheduler.state_dict(),
            'criterion_center': criterion.center.cpu(),
            'history':          history,
        }, ckpt_path)

    # Сохраняем финальный SSL backbone
    ssl_bb_path = os.path.join(save_dir, f'dinov2_ssl_{label}.pt')
    torch.save(dino_model.student_backbone.state_dict(), ssl_bb_path)
    with open(os.path.join(save_dir, f'history_ssl_{label}.json'), 'w') as f:
        json.dump(history, f)
    print(f'\nSSL завершён. Backbone сохранён: {ssl_bb_path}')

    return dino_model.student_backbone, history


print('train_ssl_dino определена.')

train_ssl_dino определена.


## 10. Запуск SSL (60 эпох)

In [13]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM свободно: {free/1e9:.1f} / {total/1e9:.1f} Гб')

print('='*65)
print(f'Эксп.18 v4 — SSL DINO фаза')
print(f'  batch={SSL_BATCH_SIZE}, epochs={SSL_EPOCHS}')
print(f'  Метод: self-distillation (LightlySSL DINO)')
print(f'  Нет негативных примеров — работает с малым batch')
print('='*65)

ssl_t0 = time.perf_counter()
dinov2_ssl_v4, ssl_hist_v4 = train_ssl_dino(all_train_imgs, TRAIN_IMG_DIR)
ssl_time = time.perf_counter() - ssl_t0

print(f'\nВремя SSL-фазы: {ssl_time/60:.1f} мин. '
      f'({ssl_time/SSL_EPOCHS:.1f} сек./эпоха)')

VRAM свободно: 7.0 / 8.5 Гб
Эксп.18 v4 — SSL DINO фаза
  batch=4, epochs=60
  Метод: self-distillation (LightlySSL DINO)
  Нет негативных примеров — работает с малым batch
DINO SSL: 60 эп., batch=4
  Изображений:      12,568
  Out dim:          2048
  Teacher momentum: 0.996 → 1.0 (косинус)
  Teacher temp warmup: 5 эп.
  Backbone заморожено: 58.2М  |  обучается: 28.4М
  Батчей/эпоху: 3142


D:\VKR\VKR\.venv\lib\site-packages\torch\nn\utils\weight_norm.py:144: FutureWarning: `torch.nn.utils.weight_norm` is deprecated in favor of `torch.nn.utils.parametrizations.weight_norm`.
  WeightNorm.apply(module, name, dim)


  Найден чекпоинт: D:/VKR/VKR/dino_heads\ssl_checkpoint_exp18v4.pt — продолжаем обучение
  Resume с эпохи 4


KeyboardInterrupt: 

## 11. SSL convergence — анализ DINO loss

In [ ]:
# Загружаем истории предыдущих версий для сравнения
prev_ssl = {}
for name, fname in [
    ('v1 SimCLR (batch=2, 15 эп.)', 'history_ssl_exp18.json'),
    ('v2 SimCLR (batch=4, 30 эп.)', 'history_ssl_exp18v2.json'),
]:
    p = os.path.join(save_dir, fname)
    if os.path.exists(p):
        with open(p) as f: prev_ssl[name] = json.load(f)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors = {'v1 SimCLR (batch=2, 15 эп.)': '#9E9E9E',
          'v2 SimCLR (batch=4, 30 эп.)': '#FF7043'}
for name, hist in prev_ssl.items():
    axes[0].plot(hist['ssl_loss'], color=colors.get(name, 'gray'),
                 lw=1.5, alpha=0.7, label=f'{name}  min={min(hist["ssl_loss"]):.4f}')

axes[0].plot(ssl_hist_v4['ssl_loss'], color='#1565C0', lw=2.5,
             label=f'v4 DINO (batch={SSL_BATCH_SIZE}, {SSL_EPOCHS} эп.)  '
                   f'min={min(ssl_hist_v4["ssl_loss"]):.4f}')
axes[0].set_title('SSL Loss (все версии)', fontsize=13)
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('Loss')
axes[0].legend(fontsize=9); axes[0].grid(True, alpha=0.3)

axes[1].plot(ssl_hist_v4['ssl_loss'], color='#1565C0', lw=2)
axes[1].plot(ssl_hist_v4['momentum'], color='#E53935', lw=1.5,
             alpha=0.7, label='Teacher momentum')
axes[1].set_title('DINO v4: loss и momentum', fontsize=13)
axes[1].set_xlabel('Эпоха'); axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'ssl_convergence_v4.png'), dpi=150)
plt.show()

## 12. Fine-tuning (точный пайплайн Эксп.17 с DINO SSL backbone)

In [ ]:
def train_finetune_with_ssl(ssl_backbone, train_ids, val_ids, df,
                             synth_pairs=None, n_epochs=EPOCHS,
                             label='ft_exp18v4', checkpoint_every=25):
    """Fine-tuning идентичен Эксп.17, но backbone — после SSL DINO."""
    for p in ssl_backbone.parameters(): p.requires_grad = False
    for i in range(n_blocks - N_UNFREEZE, n_blocks):
        for p in ssl_backbone.blocks[i].parameters(): p.requires_grad = True
    for p in ssl_backbone.norm.parameters(): p.requires_grad = True

    head = SegHeadDPT().to(DEVICE)

    bb_groups  = make_backbone_param_groups(ssl_backbone)
    head_group = [{'params': list(head.parameters()),
                   'lr': LR_HEAD, 'weight_decay': 1e-4, 'name': 'head'}]
    opt    = optim.AdamW(bb_groups + head_group)
    sched  = optim.lr_scheduler.CosineAnnealingLR(opt, T_max=n_epochs)
    scaler = torch.amp.GradScaler('cuda') if USE_AMP else None

    print('Веса классов:')
    cw   = compute_class_weights(train_ids, df)
    cce  = nn.CrossEntropyLoss(weight=cw, reduction='none')
    clov = LovászSoftmax()

    if synth_pairs is not None:
        tds = SteelSegDatasetWithSynth(
            train_ids, TRAIN_IMG_DIR, df, train_jt, synth_pairs)
    else:
        tds = SteelSegDataset(train_ids, TRAIN_IMG_DIR, df, train_jt)
    vds = SteelSegDataset(val_ids, TRAIN_IMG_DIR, df, val_jt)
    tdl = DataLoader(tds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
    vdl = DataLoader(vds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

    iou  = JaccardIndex(task='multiclass', num_classes=NUM_CLASSES,
                        average='none').to(DEVICE)
    hist = {'train_loss': [], 'val_miou': [], 'val_iou_per_class': []}
    best = 0.0
    best_state_head = None
    best_state_bb   = None

    print(f'Обучающих: {len(tds)}  |  Валидационных: {len(vds)}')
    print(f'AMP: {USE_AMP}  |  Batch: {BATCH_SIZE}  |  Батчей/эпоху: {len(tdl)}')

    for ep in range(1, n_epochs + 1):
        ssl_backbone.eval()
        for i in range(n_blocks - N_UNFREEZE, n_blocks):
            ssl_backbone.blocks[i].train()
        ssl_backbone.norm.train()
        head.train()

        tl = 0.0
        for imgs, masks, is_synth in tdl:
            imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
            opt.zero_grad()

            if scaler:
                with torch.amp.autocast('cuda'):
                    feats = ssl_backbone.get_intermediate_layers(
                        imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                    lg = head(feats)
                    px_loss = cce(lg, masks)
                    w_batch = torch.where(
                        is_synth.bool().to(DEVICE)[:, None, None],
                        torch.full_like(px_loss, SYNTH_WEIGHT),
                        torch.ones_like(px_loss))
                    loss = (px_loss * w_batch).mean() + clov(lg, masks)
                scaler.scale(loss).backward()
                scaler.step(opt)
                scaler.update()
            else:
                feats = ssl_backbone.get_intermediate_layers(
                    imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                lg = head(feats)
                px_loss = cce(lg, masks)
                w_batch = torch.where(
                    is_synth.bool().to(DEVICE)[:, None, None],
                    torch.full_like(px_loss, SYNTH_WEIGHT),
                    torch.ones_like(px_loss))
                loss = (px_loss * w_batch).mean() + clov(lg, masks)
                loss.backward()
                opt.step()

            tl += loss.item()
        sched.step()

        ssl_backbone.eval(); head.eval(); iou.reset()
        with torch.no_grad():
            for imgs, masks, _ in vdl:
                imgs, masks = imgs.to(DEVICE), masks.to(DEVICE)
                if USE_AMP:
                    with torch.amp.autocast('cuda'):
                        feats = ssl_backbone.get_intermediate_layers(
                            imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                        preds = head(feats).argmax(1)
                else:
                    feats = ssl_backbone.get_intermediate_layers(
                        imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                    preds = head(feats).argmax(1)
                iou.update(preds, masks)

        ipc = iou.compute().cpu().numpy()
        mi  = float(ipc.mean())
        hist['train_loss'].append(tl / len(tdl))
        hist['val_miou'].append(mi)
        hist['val_iou_per_class'].append(ipc.tolist())

        if mi > best:
            best = mi
            best_state_head = {k: v.clone() for k, v in head.state_dict().items()}
            best_state_bb   = {k: v.clone() for k, v in ssl_backbone.state_dict().items()}

        if ep % 10 == 0 or ep == 1:
            s = '  '.join([f'cls{i}:{v:.3f}' for i, v in enumerate(ipc)])
            print(f'[{label}] Эп {ep:3d}/{n_epochs} | '
                  f'Loss:{tl/len(tdl):.4f} | mIoU:{mi:.4f} | {s}')

        if ep % checkpoint_every == 0:
            ckpt_path = os.path.join(save_dir, f'ckpt_{label}_ep{ep}.pt')
            torch.save({'epoch': ep, 'head_state': head.state_dict(),
                        'backbone_state': ssl_backbone.state_dict(),
                        'history': hist, 'best_miou': best}, ckpt_path)
            print(f'  Чекпоинт: {ckpt_path}')

    head.load_state_dict(best_state_head)
    ssl_backbone.load_state_dict(best_state_bb)
    print(f'\n  -> Лучший mIoU: {best:.4f}')
    return head, hist


print('Fine-tuning функция определена.')

## 13. Запуск Fine-tuning

In [ ]:
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f'VRAM свободно: {free/1e9:.1f} / {total/1e9:.1f} Гб')

print('='*70)
print(f'Эксп.18 v4 — Fine-tuning ({EPOCHS} эпох, пайплайн Эксп.17)')
print(f'  Backbone: DINO SSL-адаптированный ({SSL_EPOCHS} эп.)')
print(f'  Данные: {len(selected_ids)} реал. + {len(synth_pairs)} синт.')
print(f'  Loss: CE(weighted) + Lovász | AMP: {USE_AMP}')
print('='*70)

ft_t0 = time.perf_counter()
model_18v4, history_18v4 = train_finetune_with_ssl(
    dinov2_ssl_v4, selected_ids, test_ids, train_df,
    synth_pairs=synth_pairs, label='ft_exp18v4'
)
ft_time = time.perf_counter() - ft_t0

print(f'\nВремя fine-tuning: {ft_time/60:.1f} мин. '
      f'({ft_time/EPOCHS:.1f} сек./эпоха)')
print(f'Суммарное время (SSL + FT): {(ssl_time + ft_time)/60:.1f} мин.')

## 14. Сравнение: Эксп.17, Эксп.18 v1/v2/v4

In [ ]:
histories = {}
for name, fname in [
    ('Эксп.17 (без SSL)',           'history_exp17.json'),
    ('Эксп.18 v1 (SimCLR, b=2)',    'history_exp18.json'),
    ('Эксп.18 v2 (SimCLR, b=4)',    'history_exp18v2.json'),
]:
    p = os.path.join(save_dir, fname)
    if os.path.exists(p):
        with open(p) as f: histories[name] = json.load(f)
histories['Эксп.18 v4 (DINO SSL)'] = history_18v4

names_ordered = [
    'Эксп.17 (без SSL)',
    'Эксп.18 v1 (SimCLR, b=2)',
    'Эксп.18 v2 (SimCLR, b=4)',
    'Эксп.18 v4 (DINO SSL)',
]
colors = ['#2196F3', '#9E9E9E', '#FF7043', '#1B5E20']

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for name, color in zip(names_ordered, colors):
    if name not in histories: continue
    h  = histories[name]
    mi = max(h['val_miou'])
    ep = int(np.argmax(h['val_miou'])) + 1
    lw = 2.5 if 'v4' in name else 1.5
    axes[0].plot(h['val_miou'], color=color, lw=lw,
                 label=f'{name}\nbest={mi:.4f} (эп.{ep})')

axes[0].set_title('Val mIoU по эпохам', fontsize=13)
axes[0].set_xlabel('Эпоха'); axes[0].set_ylabel('mIoU')
axes[0].legend(fontsize=8, loc='lower right'); axes[0].grid(True, alpha=0.3)

# Bar chart — лучший mIoU
best_miou = {n: max(h['val_miou']) for n, h in histories.items()
             if n in names_ordered}
bar_names  = [n.replace('Эксп.', 'Э.') for n in best_miou.keys()]
bar_vals   = list(best_miou.values())
bar_colors = [c for n, c in zip(names_ordered, colors) if n in best_miou]
bars = axes[1].bar(range(len(bar_vals)), bar_vals, color=bar_colors, width=0.5)
for b, v in zip(bars, bar_vals):
    axes[1].text(b.get_x() + b.get_width()/2, v + 0.002, f'{v:.4f}',
                 ha='center', va='bottom', fontsize=10)
axes[1].set_xticks(range(len(bar_names))); axes[1].set_xticklabels(bar_names, fontsize=9)
axes[1].set_title('Лучший Val mIoU', fontsize=13)
axes[1].set_ylim(0.60, max(bar_vals) + 0.02); axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig(os.path.join(save_dir, 'comparison_v4.png'), dpi=150)
plt.show()

print('\nИтоговое сравнение:')
base17 = max(histories.get('Эксп.17 (без SSL)', {}).get('val_miou', [0]))
for name in names_ordered:
    if name not in histories: continue
    m = max(histories[name]['val_miou'])
    delta = m - base17
    print(f'  {name:40s}: {m:.4f}  ({delta:+.4f} к Эксп.17)')

## 15. Сохранение

In [ ]:
torch.save(model_18v4.state_dict(),
           os.path.join(save_dir, 'model_exp18v4.pt'))
torch.save(dinov2_ssl_v4.state_dict(),
           os.path.join(save_dir, 'dinov2_exp18v4.pt'))
with open(os.path.join(save_dir, 'history_exp18v4.json'), 'w') as f:
    json.dump(history_18v4, f)

m_v4 = max(history_18v4['val_miou'])
e_v4 = int(np.argmax(history_18v4['val_miou'])) + 1
i_v4 = history_18v4['val_iou_per_class'][e_v4 - 1]

print('Сохранено:')
print('  model_exp18v4.pt      — голова DPT')
print('  dinov2_exp18v4.pt     — backbone (после DINO SSL + fine-tuning)')
print('  dinov2_ssl_exp18v4.pt — backbone только после DINO SSL')
print('  history_exp18v4.json  — история fine-tuning')
print('  history_ssl_exp18v4.json — история SSL')
print(f'\nЛучший mIoU Эксп.18 v4: {m_v4:.4f}  (эпоха {e_v4})')
print(f'IoU по классам: {[f"{v:.3f}" for v in i_v4]}')

base17 = max(histories.get('Эксп.17 (без SSL)', {}).get('val_miou', [0]))
if base17 > 0:
    print(f'Прирост к Эксп.17: {m_v4 - base17:+.4f} ({(m_v4-base17)/base17*100:+.2f}%)')

## 16. Dice-метрика (на val)

Считаем TP/FP/FN по всей val-выборке (1 334 изображения).  
Формула: $\text{Dice}_c = \frac{2 \cdot TP_c}{2 \cdot TP_c + FP_c + FN_c}$

In [ ]:
class DiceAccumulator:
    def __init__(self, num_classes):
        self.num_classes = num_classes
        self.tp = torch.zeros(num_classes)
        self.fp = torch.zeros(num_classes)
        self.fn = torch.zeros(num_classes)

    def update(self, preds, targets):
        preds   = preds.cpu()
        targets = targets.cpu()
        for c in range(self.num_classes):
            pred_c   = (preds   == c)
            target_c = (targets == c)
            self.tp[c] += ( pred_c &  target_c).sum().float()
            self.fp[c] += ( pred_c & ~target_c).sum().float()
            self.fn[c] += (~pred_c &  target_c).sum().float()

    def compute(self):
        dice = 2 * self.tp / (2 * self.tp + self.fp + self.fn + 1e-8)
        return dice.numpy(), float(dice.mean())


dice_acc = DiceAccumulator(NUM_CLASSES)
vds_full = SteelSegDataset(test_ids, TRAIN_IMG_DIR, train_df, val_jt)
vdl_full = DataLoader(vds_full, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

dinov2_ssl_v4.eval(); model_18v4.eval()
with torch.no_grad():
    for imgs, masks, _ in tqdm(vdl_full, desc='Dice на val'):
        imgs = imgs.to(DEVICE)
        if USE_AMP:
            with torch.amp.autocast('cuda'):
                feats = dinov2_ssl_v4.get_intermediate_layers(
                    imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
                preds = model_18v4(feats).argmax(1).cpu()
        else:
            feats = dinov2_ssl_v4.get_intermediate_layers(
                imgs, n=INTERMEDIATE_LAYERS, return_class_token=False)
            preds = model_18v4(feats).argmax(1).cpu()
        dice_acc.update(preds, masks)

dice_per_class, mean_dice = dice_acc.compute()
cls_names = ['Фон', 'Дефект 1', 'Дефект 2', 'Дефект 3', 'Дефект 4']

print(f'\nDice на val ({len(test_ids)} изображений):')
for cn, d in zip(cls_names, dice_per_class):
    print(f'  {cn:12s}: {d:.4f}')
print(f'  Mean Dice:    {mean_dice:.4f}')
print(f'  Best mIoU:    {m_v4:.4f}')

## 17. Метрики эффективности модели

In [ ]:
n_bb    = sum(p.numel() for p in dinov2_ssl_v4.parameters()) / 1e6
n_hd    = sum(p.numel() for p in model_18v4.parameters())    / 1e6
n_total = n_bb + n_hd

head_path = os.path.join(save_dir, 'model_exp18v4.pt')
bb_path   = os.path.join(save_dir, 'dinov2_exp18v4.pt')
head_mb   = os.path.getsize(head_path) / (1024**2) if os.path.exists(head_path) else 0
bb_mb     = os.path.getsize(bb_path)   / (1024**2) if os.path.exists(bb_path)   else 0

# Скорость инференса (среднее по 20 прогонам)
sample_img = next(iter(
    DataLoader(SteelSegDataset([test_ids[0]], TRAIN_IMG_DIR, train_df, val_jt),
               batch_size=1)))[0].to(DEVICE)

dinov2_ssl_v4.eval(); model_18v4.eval()
with torch.no_grad():
    for _ in range(5):   # прогрев
        feats = dinov2_ssl_v4.get_intermediate_layers(
            sample_img, n=INTERMEDIATE_LAYERS, return_class_token=False)
        _ = model_18v4(feats)

    if torch.cuda.is_available(): torch.cuda.synchronize()
    t0 = time.perf_counter()
    N_RUNS = 20
    for _ in range(N_RUNS):
        feats = dinov2_ssl_v4.get_intermediate_layers(
            sample_img, n=INTERMEDIATE_LAYERS, return_class_token=False)
        _ = model_18v4(feats)
    if torch.cuda.is_available(): torch.cuda.synchronize()
    inf_time = (time.perf_counter() - t0) / N_RUNS

if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
    with torch.no_grad():
        feats = dinov2_ssl_v4.get_intermediate_layers(
            sample_img, n=INTERMEDIATE_LAYERS, return_class_token=False)
        _ = model_18v4(feats)
    vram_mb = torch.cuda.max_memory_allocated() / (1024**2)
else:
    vram_mb = 0

print('Метрики эффективности Эксп.18 v4:')
print(f'  Параметры: {n_total:.1f}М (backbone {n_bb:.1f}М + head {n_hd:.1f}М)')
print(f'  Размер весов: {head_mb+bb_mb:.1f} МБ (head {head_mb:.1f} + bb {bb_mb:.1f})')
print(f'  Инференс: {inf_time:.4f} сек/изобр. ({1/inf_time:.1f} FPS)')
print(f'  VRAM на изображение: {vram_mb:.1f} МБ')
print(f'  Время обучения: SSL {ssl_time/60:.1f} мин. + FT {ft_time/60:.1f} мин.')